In [12]:
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler

In [13]:
data=r"C:\Users\Mahakaal\Documents\churnsense\data\processed\Telco-Customer-Churn dataset_eda_complete.csv"
df=pd.read_csv(data)
df.head().T

,0,1,2,3,4
seniorcitizen,0,0,0,0,0
partner,1,0,0,0,0
tenure,1,34,2,45,2
phoneservice,0,1,1,0,1
internetservice,dsl,dsl,dsl,dsl,fiber_optic
onlinesecurity,no,yes,yes,yes,no
onlinebackup,yes,no,yes,no,no
deviceprotection,no,yes,no,yes,no
techsupport,no,no,no,yes,no
streamingtv,no,no,no,no,no


In [14]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['churn'])
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42, stratify=df_full_train['churn'])

y_train = df_train['churn']
y_val = df_val['churn']
y_test = df_test['churn']

x_train = df_train.drop(columns=['churn'])
x_val = df_val.drop(columns=['churn'])
x_test = df_test.drop(columns=['churn'])

print(f"Train size: {len(x_train)}, Val size: {len(x_val)}, Test size: {len(x_test)}")

Train size: 4225, Val size: 1409, Test size: 1409


In [15]:
class ChurnModelPipeline:
    def __init__(self, c_val=1.0, max_iter=1000, random_state=42):
        self.c_val = c_val
        self.max_iter = max_iter
        self.random_state = random_state
        self.dv = DictVectorizer(sparse=False)
        self.scaler = StandardScaler()
        self.model = LogisticRegression(
            C=self.c_val,
            max_iter=self.max_iter,
            random_state=self.random_state
        )
        self.optimal_threshold = 0.5

    def fit(self, df_x, y):
        records = df_x.to_dict(orient='records')
        x_vec = self.dv.fit_transform(records)
        x_scaled = self.scaler.fit_transform(x_vec)
        self.model.fit(x_scaled, y)
        return self

    def predict_proba(self, df_x):
        records = df_x.to_dict(orient='records')
        x_vec = self.dv.transform(records)
        x_scaled = self.scaler.transform(x_vec)
        return self.model.predict_proba(x_scaled)[:, 1]

    def predict(self, df_x, threshold=None):
        t = threshold if threshold is not None else self.optimal_threshold
        probs = self.predict_proba(df_x)
        return (probs >= t).astype(int)

In [16]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)
auc_scores = []

for train_idx, val_idx in kf.split(df_full_train):
    fold_train = df_full_train.iloc[train_idx]
    fold_val = df_full_train.iloc[val_idx]

    pipe = ChurnModelPipeline()
    pipe.fit(fold_train.drop(columns=['churn']), fold_train['churn'])
    preds = pipe.predict_proba(fold_val.drop(columns=['churn']))
    auc_scores.append(roc_auc_score(fold_val['churn'], preds))

print(f"Mean CV ROC-AUC: {np.mean(auc_scores):.4f} +/- {np.std(auc_scores):.4f}")

Mean CV ROC-AUC: 0.8456 +/- 0.0161


In [17]:
pipeline = ChurnModelPipeline()
pipeline.fit(x_train, y_train)

val_probs = pipeline.predict_proba(x_val)

thresholds = np.linspace(0.01, 0.99, 100)
scores = [
    (t, f1_score(y_val, (val_probs >= t).astype(int)), accuracy_score(y_val, (val_probs >= t).astype(int)))
    for t in thresholds
]
scores.sort(key=lambda item: item[1], reverse=True)
best_threshold, best_f1, best_acc = scores[0]
pipeline.optimal_threshold = best_threshold

print(f"Optimal Threshold: {best_threshold:.2f}")
print(f"Validation F1: {best_f1:.4f} | Accuracy: {best_acc:.4f} | ROC-AUC: {roc_auc_score(y_val, val_probs):.4f}")

Optimal Threshold: 0.30
Validation F1: 0.6311 | Accuracy: 0.7644 | ROC-AUC: 0.8354


In [18]:
test_probs = pipeline.predict_proba(x_test)
test_preds = pipeline.predict(x_test)

print(f"Test ROC-AUC: {roc_auc_score(y_test, test_probs):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_preds))
print("\nClassification Report:")
print(classification_report(y_test, test_preds, digits=4))

Test ROC-AUC: 0.8440

Confusion Matrix:
[[777 258]
 [ 87 287]]

Classification Report:
              precision    recall  f1-score   support

           0     0.8993    0.7507    0.8183      1035
           1     0.5266    0.7674    0.6246       374

    accuracy                         0.7551      1409
   macro avg     0.7130    0.7591    0.7215      1409
weighted avg     0.8004    0.7551    0.7669      1409



In [19]:
final_pipeline = ChurnModelPipeline()
final_pipeline.fit(df_full_train.drop(columns=['churn']), df_full_train['churn'])
final_pipeline.optimal_threshold = best_threshold

output_model_path = "model.bin"
with open(output_model_path, "wb") as f_out:
    pickle.dump(final_pipeline, f_out)